In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, r2_score,
    classification_report,
)

sns.set_theme(style="whitegrid")
SEED = 42

In [ ]:
# Load
df = pd.read_excel('../data/raw/Telco_customer_churn.xlsx', sheet_name='Telco_Churn')
print(f"Raw shape: {df.shape}")

# Drop columns specified by user + City (high cardinality string, not in encode list)
# + Churn Label (text version of the target — we predict Churn Value instead)
# + Total Charges (multicollinearity with Tenure Months — see ML_experiments_decisions.md)
# Latitude and Longitude are kept as continuous geographic features
cols_to_drop = [
    "Count", "Country", "State", "Lat Long",
    "Churn Score", "Churn Reason",
    "City",          # 1 129 unique string values, not in the encode list
    "Churn Label",   # text mirror of Churn Value — leaks the target
    "Total Charges", # high collinearity with Tenure Months (r=0.83, VIF>10)
]
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

print(f"Shape after drops: {df.shape}")
display(df.head())

In [ ]:
# One-hot encode categorical columns
cat_cols = [
    "Gender", "Senior Citizen", "Partner", "Dependents",
    "Phone Service", "Multiple Lines", "Internet Service",
    "Online Security", "Online Backup", "Device Protection",
    "Tech Support", "Streaming TV", "Streaming Movies",
    "Contract", "Paperless Billing", "Payment Method",
]
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Features / target split — exclude CustomerID and target
X = df.drop(columns=["CustomerID", "Churn Value"]).astype(float)
y = df["Churn Value"]

# 80 / 20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

print(f"Train shape : {X_train.shape}")
print(f"Test  shape : {X_test.shape}")
# y.mean() on a 0/1 column = proportion of churned customers
print(f"Train churn rate : {y_train.mean():.2%}")
print(f"Test  churn rate : {y_test.mean():.2%}")

# Standardize — fit only on train to avoid data leakage into test
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print("\nStandardization done — mean≈0, std≈1 on training set")

In [ ]:
def compute_metrics(y_true, y_pred, y_prob=None, model_name="Model"):
    """Return a labelled Series of classification + regression-style metrics."""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # MAPE and MRE are undefined when y_true == 0; restrict to positive class rows
    mask = y_true != 0
    if mask.sum() > 0:
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
        mre  = np.mean((y_pred[mask] - y_true[mask]) / y_true[mask]) * 100
    else:
        mape = mre = float("nan")

    metrics = {
        "Accuracy"   : accuracy_score(y_true, y_pred),
        "Precision"  : precision_score(y_true, y_pred, zero_division=0),
        "Recall"     : recall_score(y_true, y_pred, zero_division=0),
        "F1"         : f1_score(y_true, y_pred, zero_division=0),
        "R²"         : r2_score(y_true, y_pred),
        "MAPE (%)"   : round(mape, 4),
        "MRE (%)"    : round(mre, 4),
    }
    if y_prob is not None:
        metrics["AUC-ROC"] = roc_auc_score(y_true, y_prob)
        metrics["PR-AUC"]  = average_precision_score(y_true, y_prob)

    return pd.Series(metrics, name=model_name)

In [ ]:
# Correlation + VIF — numerical variables (including target)
# Uses a temporary raw load so Total Charges can be shown before its drop is applied
from statsmodels.stats.outliers_influence import variance_inflation_factor

_df_raw = pd.read_excel('../data/raw/Telco_customer_churn.xlsx', sheet_name='Telco_Churn')
_df_raw["Total Charges"] = pd.to_numeric(_df_raw["Total Charges"], errors="coerce")
_df_raw["Churn Value"] = _df_raw["Churn Value"].astype(float)

num_cols_corr = ["Tenure Months", "Monthly Charges", "Total Charges", "CLTV", "Churn Value"]

# Correlation heatmap
corr = _df_raw[num_cols_corr].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Correlation Matrix — Numerical Variables (pre-drop)")
plt.tight_layout()
plt.show()

display(corr.style.background_gradient(cmap="coolwarm", vmin=-1, vmax=1).format("{:.2f}"))

# VIF — computed on numerical feature columns before any drops
vif_cols = ["Tenure Months", "Monthly Charges", "Total Charges", "CLTV"]
vif_data = _df_raw[vif_cols].dropna().astype(float)

vif_df = pd.DataFrame({
    "Feature": vif_cols,
    "VIF": [variance_inflation_factor(vif_data.values, i) for i in range(len(vif_cols))],
}).sort_values("VIF", ascending=False).reset_index(drop=True)

print("\nVariance Inflation Factor (VIF)  —  threshold: VIF > 10 indicates problematic multicollinearity")
display(vif_df)

del _df_raw

In [ ]:
# Dummy Classifier
dummy = DummyClassifier(strategy="most_frequent", random_state=SEED)
dummy.fit(X_train_scaled, y_train)

y_pred_dummy = dummy.predict(X_test_scaled)
y_prob_dummy = dummy.predict_proba(X_test_scaled)[:, 1]

print("=== Dummy Classifier ===")
print(classification_report(y_test, y_pred_dummy, target_names=["No Churn", "Churn"]))
display(compute_metrics(y_test, y_pred_dummy, y_prob_dummy, "Dummy Classifier").to_frame())

In [ ]:
# Logistic Regression — classification threshold lowered to 0.3
# Justification: see ML_experiments_decisions.md
THRESHOLD = 0.3

lr = LogisticRegression(random_state=SEED, max_iter=10000)
lr.fit(X_train_scaled, y_train)

y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]
y_pred_lr = (y_prob_lr >= THRESHOLD).astype(int)

print("=== Logistic Regression (threshold=0.3) ===")
print(classification_report(y_test, y_pred_lr, target_names=["No Churn", "Churn"]))
display(compute_metrics(y_test, y_pred_lr, y_prob_lr, "Logistic Regression").to_frame())

In [ ]:
# SHAP values — Logistic Regression feature contributions
import shap

explainer = shap.LinearExplainer(lr, X_train_scaled, feature_perturbation="interventional")
shap_values = explainer.shap_values(X_test_scaled)

feature_names = X.columns.tolist()

# Beeswarm: direction and magnitude per prediction
plt.figure()
shap.summary_plot(shap_values, X_test_scaled, feature_names=feature_names, show=False)
plt.title("SHAP Summary — Logistic Regression (beeswarm)")
plt.tight_layout()
plt.show()

# Bar: mean absolute SHAP with value labels
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = (
    pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs_shap})
    .sort_values("mean_abs_shap", ascending=True)
)

fig, ax = plt.subplots(figsize=(10, max(6, len(feature_names) * 0.35)))
bars = ax.barh(
    shap_importance["feature"],
    shap_importance["mean_abs_shap"],
    color=sns.color_palette("viridis", len(shap_importance)),
)
for bar, val in zip(bars, shap_importance["mean_abs_shap"]):
    ax.text(
        bar.get_width() + 0.005,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.4f}",
        va="center", fontsize=8,
    )
ax.set_xlim(0, shap_importance["mean_abs_shap"].max() * 1.18)
ax.set_xlabel("mean |SHAP value|")
ax.set_title("SHAP Feature Importance — Logistic Regression (mean |SHAP|)")
plt.tight_layout()
plt.show()